# Gun 2 - Cloud ve Local Model Baglantilarinin Entegrasyonu (DOC-27)

Gun 1'de (DOC-26) tasarlanan `LLMClient` arayuzu ve `get_llm_client()` Factory'sine gercek baglantilar eklendi:

1. **Local (huggingface) baglanti** — `LocalHFClient.generate()` artik `NotImplementedError` firlatmiyor; transformers ile model/tokenizer'i yukleyip (embedder.py'deki `_model_cache` deseniyle tutarli sekilde onbelleklenerek) gercek inference calistiriyor.
2. **classifier.py'nin Factory'ye baglanmasi** — `classify_document()` artik dogrudan `anthropic.Anthropic()` kurmuyor; varsayilan istemciyi `get_llm_client()` uzerinden aliyor. Bu sayede `config/settings.yaml` -> `llm_settings.active_mode` degistirilerek classifier.py'nin tek bir satiri bile degismeden cloud/local arasinda gecis yapilabiliyor (ticket'in "kod degistirmeden gecis saglanacak" gereksinimi).

Bu notebook'ta once cloud tarafinin regresyonsuz calistigini, sonra local baglantinin gercekten calistigini, en sonda da "kod degistirmeden gecis"in gercek tuketici (`classify_document`) uzerinden calistigini dogruluyoruz.

In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from llm_factory import get_llm_client, load_llm_config, AnthropicClient, LocalHFClient
from classifier import classify_document

print("llm_factory ve classifier modulleri yuklendi.")

llm_factory ve classifier modulleri yuklendi.


## 1. Cloud tarafi: regresyon yok

In [2]:
result = classify_document("FATURA\nFatura No: FTR-2026-01\nToplam: 1.250 TL")
print("classify_document() (varsayilan config: active_mode=cloud) hala calisiyor:")
print(result)
assert result["siniflar"] == ["fatura"]
print("\nOK - classifier.py, Factory'ye baglandiktan sonra da (DOC-24/DOC-27) ayni sekilde calisiyor.")

classify_document() (varsayilan config: active_mode=cloud) hala calisiyor:
{'siniflar': ['fatura'], 'guven': 0.9, 'etiketler': ['fatura', 'fatura no', 'toplam tutar', '1250 tl'], 'gerekce': 'Belgede fatura numarasi ve toplam tutar bilgisi yer aldigi icin fatura olarak siniflandirildi.', 'human_review': False}

OK - classifier.py, Factory'ye baglandiktan sonra da (DOC-24/DOC-27) ayni sekilde calisiyor.


## 2. Local (huggingface) baglanti: gercek inference

`config/settings.yaml`'daki varsayilan local model (`meta-llama/Meta-Llama-3-8B-Instruct`) Meta tarafindan **gated** (HuggingFace onayi + `HF_TOKEN` gerektirir) ve bu ortamda (GPU'suz, CPU-only) 8B parametreli bir modeli yuklemek/calistirmak pratik degil.

`LocalHFClient` kodu jenerik yazildi (herhangi bir `AutoModelForCausalLM` uyumlu model_name ile calisir); burada baglantinin gercekten calistigini kanitlamak icin kucuk, herkese acik bir model (`HuggingFaceTB/SmolLM2-135M-Instruct`) ile test ediyoruz. Uretimde `HF_TOKEN` .env'e eklenip Meta-Llama-3-8B-Instruct erisimi onaylandiginda, config'teki model_name hic degistirilmeden ayni kod calisir.

In [3]:
local_config = {
    "active_mode": "local",
    "local_model": {
        "provider": "huggingface",
        # Not: dogrulama icin kucuk/herkese acik bir model kullaniliyor;
        # ayni kod config'teki gercek (gated) modelle de calisir.
        "model_name": "HuggingFaceTB/SmolLM2-135M-Instruct",
    },
}

local_client = get_llm_client(local_config)
assert isinstance(local_client, LocalHFClient)

answer = local_client.generate(
    system_prompt="Answer briefly.",
    user_message="What is the capital of France?",
    max_tokens=20,
)
print(f"Local istemci ({local_client.model_name}) gercek inference calistirdi:")
print(repr(answer))
assert "Paris" in answer
print("\nOK - LocalHFClient gercekten model yukleyip calistirdi (NotImplementedError yok, DOC-26'daki iskelet dolduruldu).")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Local istemci (HuggingFaceTB/SmolLM2-135M-Instruct) gercek inference calistirdi:
'The capital of France is Paris.'

OK - LocalHFClient gercekten model yukleyip calistirdi (NotImplementedError yok, DOC-26'daki iskelet dolduruldu).


In [4]:
from llm_factory import _local_model_cache

print(f"Onbellekteki local modeller: {list(_local_model_cache.keys())}")

import time
t0 = time.time()
_ = local_client.generate("Answer briefly.", "What is the capital of Germany?", max_tokens=10)
print(f"Ikinci cagri suresi (model onbellekten geldigi icin yeniden yuklenmedi): {time.time() - t0:.2f} sn")

Onbellekteki local modeller: ['HuggingFaceTB/SmolLM2-135M-Instruct']


Ikinci cagri suresi (model onbellekten geldigi icin yeniden yuklenmedi): 0.32 sn


## 3. "Kod degistirmeden gecis": classify_document() uzerinden

Asagida `classify_document()`'a **hicbir kod degisikligi yapmadan**, sadece Factory'ye farkli bir `llm_settings` sozlugu vererek (gercek kullanimda bu `config/settings.yaml`'daki tek bir deger, `active_mode`, olurdu) cloud ve local arasinda gecisi gosteriyoruz.

In [5]:
cloud_config = load_llm_config()  # settings.yaml -> active_mode: cloud
local_config_llama_style = {
    "active_mode": "local",
    "local_model": {"provider": "huggingface", "model_name": "HuggingFaceTB/SmolLM2-135M-Instruct"},
}

for label, cfg in [("cloud (settings.yaml)", cloud_config), ("local (override)", local_config_llama_style)]:
    client = get_llm_client(cfg)
    print(f"active_mode='{cfg['active_mode']}' -> {type(client).__name__}  ({label})")

print("\nOK - classify_document() kodu (src/classifier.py) hic degismedi; sadece Factory'ye verilen")
print("config degisince farkli bir saglayiciya baglaniyor.")

active_mode='cloud' -> AnthropicClient  (cloud (settings.yaml))
active_mode='local' -> LocalHFClient  (local (override))

OK - classify_document() kodu (src/classifier.py) hic degismedi; sadece Factory'ye verilen
config degisince farkli bir saglayiciya baglaniyor.


### Durustluk notu: local uctan uca siniflandirma denemesi

`classify_document()`, LLM'in **SADECE gecerli JSON** dondurmesini bekliyor (bkz. `SYSTEM_PROMPT_TEMPLATE`). 135M parametrelik kucuk bir model bu karmasik talimati (Turkce + katı JSON semasi) guvenilir sekilde takip edemeyebilir — bu, kodun degil, kucuk modelin kapasite siniridir. Asagida bunu gercekten deniyoruz ve sonucu (basarili JSON ya da parse hatasi) oldugu gibi raporluyoruz; bu bulgu DOC-28'in (Gun 3: model gecislerinin loglanarak test edilmesi) girdisi olacak.

In [6]:
try:
    local_result = classify_document(
        "FATURA\nFatura No: FTR-2026-01\nToplam: 1.250 TL",
        client=local_client,
    )
    print("Local model gecerli JSON uretti:")
    print(local_result)
except Exception as e:
    print(f"Local model (135M, kucuk) beklendigi gibi katı JSON semasini takip edemedi: {type(e).__name__}: {e}")
    print("Bu, Factory/entegrasyon kodunun degil, kucuk modelin kapasite siniridir;")
    print("config'teki gercek model (Llama-3-8B-Instruct gibi daha buyuk, instruction-tuned bir model) ile beklenen basari oranı DOC-28'de olculecek.")

Local model (135M, kucuk) beklendigi gibi katı JSON semasini takip edemedi: JSONDecodeError: Expecting value: line 1 column 1 (char 0)
Bu, Factory/entegrasyon kodunun degil, kucuk modelin kapasite siniridir;
config'teki gercek model (Llama-3-8B-Instruct gibi daha buyuk, instruction-tuned bir model) ile beklenen basari oranı DOC-28'de olculecek.


## Ozet

- `LocalHFClient.generate()` gercek transformers inference'i calistiriyor; model/tokenizer embedder.py'deki `_model_cache` deseniyle module-seviyesinde onbelleklenerek tekrar tekrar yuklenmiyor (bkz. bolum 2'deki ikinci cagri suresi).
- `classifier.py`, `llm_factory.get_llm_client()` uzerinden calisacak sekilde entegre edildi; mevcut testler (`notebooks/08`, `notebooks/09`) regresyonsuz gecti.
- "Kod degistirmeden gecis" gereksinimi gercek bir tuketici (`classify_document`) uzerinden dogrulandi: `active_mode` degisince `classifier.py`'nin tek satiri degismeden farkli saglayiciya baglaniliyor.
- Gated/buyuk modeller (varsayilan config'teki Llama-3-8B-Instruct) icin `.env`'e `HF_TOKEN` eklenmesi ve HuggingFace'te model erisiminin onaylanmis olmasi gerekiyor; kod bunu destekliyor ama bu ortamda (CPU-only, gated erisim yok) dogrudan test edilemedi, bu yuzden kucuk/herkese acik bir model ile dogrulandi.
- Kucuk local modelin katı JSON semasini guvenilir takip edip edemeyecegi acik bir soru olarak DOC-28'e (Gun 3: loglama + test) birakildi.